<img align="left" width="30%" src="https://www.upv.es/entidades/etsinf/wp-content/uploads/2024/06/logo_etsinf_valenciano_grisUPV.svg"/> <img width="30%" src="https://www.upv.es/perfiles/pas-pdi/imagenes/marca_UPV_principal_negro150.png" align="right"/>

# Optimization - Laboratory 1: Solving linear programming models with ORTools

This is our first laboratory and it will serve as an introduction to work with ORTools, an optimization suite that covers several types of optimization problems. It assumes that you are somehow proficient with Python, as well as the Notebook-based development environment. In this practice, we will focus on solving simple linear programming models that contain a handful of variables and constraints.

# Learning outcomes

In this laboratory we will work on the following learning outcomes:

* Importing necessary variables in ORTools to solve linear programming scenarios.
* Create a linear programming solver instance to solve a problem instance.
* Manually creating continuous decision variables using ORTools.
* Manually creating and configuring an objective function using ORTools.
* Manually creating and configuring constraints using ORTools.
* Using ORTools to solve and analyze a linear programming scenario.

# Introduction to Google ORTools


One of the optimization suites available in Python for programming and solving optimization problems is Google ORTools. As its name indicates, this library is developed by Google as an open source project (code available [here](https://github.com/google/or-tools)). More information about the project can be found at the following [web](https://developers.google.com/optimization).

<figure style="text-align:center">
  <center>
  <img src="https://developers.google.com/optimization/images/orLogo_72.png"/>
  <figcaption align="center">Google ORTools Logo</figcaption>
  </center>
</figure>

In the first unit we mentioned different alternatives when implementing optimization models via a solver:
* Specialized **high level languages** like AMPL and GAMS.
* Generic APIs that cover several solvers
* Specific APIs for an optimization solver

Google ORTools is an optimization suite that falls under the second category. Although it incorporates several solvers natively, it has a generic interface that allows the developer to use both native solvers and other solvers developed by other parties (some of them commercial). Although we will work with the Python API, the native API is developed in C++. In fact, the Python API is really a wrapper for the API developed in C++, and many of the naming conventions followed for functions and methods in the Python API follow the C++ naming conventions. Following these conventions in both APIs seeks to ease the adaptation from one API to the other.

Among the types of optimization problems that can be solved with Google ORTools are:

*   Linear programming (LP)
*   Integer Linear Programming (ILP) and Binary Linear Programming
*   Constraint satisfaction problems
*   Assignment problems
*   Routing and transportation problems
*   The knapsack problem and its variants
*   Network and flow optimization problems
*   Scheduling problems

Despite its novelty, ORTools has won several academic and international competitions for constrained optimization problems. Therefore, it is quite a promising optimization suite.

In this first laboratory we are going to focus on solving linear programming (LP) problems. Recall that in linear programming problems, we can highlight some relevant features:

*    The **objective function** is a **linear combination** of **decision variables** and parameters.
*    **Constraints** are expressed by means of inequalities ($\leq, \geq, =$) that are a **linear combination** of **decision variables** and **parameters**
*    The **domain** of decision variables is **real-valued** and infinite (i.e., $X_i \in \mathbb{R}$).

By default, Google ORTools is not installed as part of the Google Colab environment, nor is it installed for default Python installations. Therefore, we will first install ORTools for Python with the *pip install* command. Run the following command in a terminal or on this notebook itself, provided that you have not previously installed ORTools before in this session/local machine. Note that, in Google Colab virtual machines, it is necessary to do this every time we restart the notebook.




In [ ]:
!pip install ortools

# Google Linear Optimizer (Glop)

As mentioned, the Google ORTools optimization suite has several solvers and algorithms that are natively installed with the optimization suite. It also provides an API for interaction with third-party solvers such as SCIP and Gurobi.

In the case of linear programming (LP) problems, ORTools has a native solver capable of solving this type of problems. In addition to this, it also provides an API to interact with other linear programming solvers such as *GLPK*, *CLP*, *CPLEX* or *Gurobi*. In this first laboratory we are going to work with the native solver provided by the suite, *Google Linear Optimizer* or also known as *GLOP*. Like the suite itself, *GLOP* is an open source project whose free code is available at the following [web](https://github.com/google/or-tools/tree/stable/ortools/glop).

This solver is installed off-the-shelf with ORTools. Thus, it does not require additional configuration. GLOP is capable of solving a wide variety of linear programming problems and it has many features that facilitate the modeling and development of linear programming scenarios.

In the next section we describ e how we can use GLOP to program and solve a simple linear programming scenario.

# Solving our first linear programming scenarios with GLOP

In this tutorial, we are going to solve a simple linear programming scenario. The description of the scenario is as follows:

> *In a paint factory two different types of paints are created: A and B. Each of these types of paint has its own manufacturing requirements. Therefore, the manufacturing process for both is not homogeneous. Specifically, each hundred liter of paint A requires 2 hours of mixing, while each hundred liter of paint B requires 4 hours of mixing. In addition to this process, each paint also requires a pigmentation process. For every hundred liters of paint A, 3 hours of pigmentation is required, and 1 hour of pigmentation is required for every hundred liters of paint B. Per day, we can only use the mixing machine for 11 hours, while the pigmenting machine is limited to 9 hours per day. If the expected profit per hundred liters produced is 6 (in thousands of euros) for paint A and 2.1 (in thousands of euros) for paint B, how many hundreds of liters of each type of paint should be produced per day?

At this point in the course, you should have no problem modeling this simple optimization problem. Before moving on to the next section, try to model the problem manually.

## 1. Modelling the scenario

$\max \;\;Z=6 A + 2.1 B$

subject to:

$mixing: 2 A + 4 B \leq 11$

$pigmentation: 3A + 1 B \leq 9 $

$A, B \geq 0, \;\; A,B \in \mathbb{R}$


## 2. Importing the ORTools wrapper for linear models

When working with a notebook, or when programming a Python script where aim to use GLOP to solve a linear programming problem, we first need to import *pywraplp*. This is the ORTools module that contains the functions and classes that will allow us to model the linear programming scenario itself, as well as to interact with the solver we are working with to solve the problem.

This module is located inside the ORTools suite and inside the *linear_solver* module. Therefore, to bring it to our notebook/script we will have to use an import instruction as the following one:




In [ ]:
from ortools.linear_solver import pywraplp

Once *pywraplp* is imported, we can interact with the functions and classes defined within to solve optimization problems.

## 3. Creating an instance of the Solver class

In order to model and solve a linear optimization problem, we need to instantiate the *Solver* class. This will generate an object of that class, which we will use to model the problem (e.g., create variables, constraints, etc.) and solve it. As a general rule, we will need to create an instance of this object for each linear optimization problem that we want to solve (*although there are ways to reuse the same object using the Clear method*).

In order to instantiate an object of the *Solver* class, we call the static method *CreateSolver* that is defined inside the class. This method takes one parameter: a the specific solver type that we want to use. In the following example, we can see how we create an instance of the class *Solver* by means of the method *CreateSolver*, and we specify that we want to use the *GLOP* solver type.

In [ ]:
solver = pywraplp.Solver.CreateSolver('GLOP')

## 4. Creating decision variables

The next natural step in modeling with ORTools is defining the decision variables. Remember that  **decision variables** represent those aspects over which we **have decision power** and can, therefore, define their final value. In the example we have presented above, as a factory, we can decide on how many hundreds of liters of paint A and paint B we manufacture per day.

Both decision variables are of real type, since we can manufacture fractional amounts of liters of paint (e.g., 0.6, 0.7, 1.3, etc.). In both cases, we can never manufacture negative quantities of paint, so the minimum value will be zero. Hypothetically, we can manufacture an arbitrary number of liters of paint (as long as the constraints of the problem allow us to do so), so we do not have a maximum value either. Therefore, both variables can be modeled as:

$A,B\geq 0 \;\; A,B \in \mathbb{R}$

In ORTools, decision variables are associated to a specific instance of the *Solver* class, so their creation is done through methods associated to objects of that class. In order to create a real-valued decision variable ($X_i \in \mathbb{R}$), we use the *NumVar* method. When we create a variable, we first define its *lower bound* or the minimum value that the variable can take (i.e., 0 in our case) and its *upper bound* or the maximum value that the variable can take. In the case of both A and B, they do not have an upper bound so we will represent this value with infinity (represented internally by the *infinity* method). Optionally, we can provide an internal name to each of these variables with the optional parameter *name*.

In the following example we create variables A and B, set their lower bound (i.e., zero), their upper bound (i.e., infinity), and give them corresponding internal names. Run the following example to define each of the variables.

In [ ]:
A = solver.NumVar(0, solver.infinity(), name='A')
B = solver.NumVar(0, solver.infinity(), name='B')

## 5. Creting constraints

Once all the decision variables have been defined, we can define the **constraints** of the scenario. It is worth remembering that, in the case of linear programming, these are specified as mathematical **inequalities** and they are a **linear combination** of decision variables and parameters. For our simple example, we defined the following constraints:

$mixing: 2 A + 4 B \leq 11$

$pigmentation: 3A + 1 B \leq 9 $

As both constraints just have an explicit upper bound, both constraints can be seen as mathematically equivalent to:

$mixing: -\infty \leq 2 A + 4 B \leq 11$

$pigmentation: -\infty \leq 3A + 1 B \leq 9 $

To define constraints in ORTools, we must associate them to a specific solver instance and, therefore, to a specific optimization scenario. We call the *Constraint* method to create a linear type constraint. When creating a constraint we have to specify both the lower and the upper bound of the constraint, as well as a name for the constraint.

In the following example we create the constraint *mixing* with its corresponding lower and upper dimensions.

In [ ]:
c_mixing = solver.Constraint(-solver.infinity(), 11, "mixing")

Once the instance of  *Constraint* has been created, we can set the coefficient that accompanies each of the decision variables in the constraint. To do this, we use the method *SetCoefficient* defined on the constraint type object. We need to provide the specific decision variable whose coefficient we want to adjust and the coefficient itself.

In the following example we will take the *c_mixing* object (i.e., where we have stored our mixing constraint) and call the *SetCoefficient* method to set both the coefficient of A to 2, and the coefficient of B to 4.

**Reminder:** $mixing: 2 A + 4 B \leq 11$.

In [ ]:
c_mixing.SetCoefficient(A, 2)
c_mixing.SetCoefficient(B, 4)

If we do not set the coefficient for a decision variable in a constraint (i.e., using *SetCoefficient*), its default coefficient will be zero. This is equivalent to specifying that the decision variable does not participate in the constraint.

Once we have defined the *mixing* constraint, define the *pigmentation* constraint in a similar way. In this case we have decided **NOT** to give the constraint an internal name.

**Reminder:** $pigmentation: 3A + 1 B \leq 9 $

In [ ]:
c_pigmentation = solver.Constraint(-solver.infinity(), 9)
c_pigmentation.SetCoefficient(A, 3)
c_pigmentation.SetCoefficient(B, 1)

## 6. Creating the objective function

One of the last steps is creating the **objective function**. In linear programming, the objective function is a **linear combination** of the decision variables. This function should be minimized or maximized depending on the particular problem to be solved.

In our case, the objective function seeks to maximize the profits of the manufacturing company (i.e., the more profits the better). For every hundred liters manufactured of product A we generate 6 (thousands of euros) of profit, while for product B it is 2.1 (thousands of euros).

$\max \;\;Z=6 A + 2.1 B$

Similarly to how we have generated constraints and decision variables, we can create objective functions by calling the *Objective* method in our solver instance. This method takes no parameters.

Once we have created the object that represents the objective of the problem, we can set the coefficient of each decision variable with the *SetCoefficient* method, as we did with constraints. Finally, we can determine whether the objective should be maximized or minimized using the *SetMaximization* or *SetMinimization* methods.

In the following code snippet, we generate the objective of the problem, adjust the coefficients for variables A and B, and define that we want to maximize the objective.

In [ ]:
objective = solver.Objective()
objective.SetCoefficient(A, 6)
objective.SetCoefficient(B, 2.1)
objective.SetMaximization()

## 7. Solving the problem

Once we have finished modeling the problem, we can use our solver instance to solve it. To do so, we need to invoke the *Solve* method. The process of solving the problem can take a matter of milliseconds, seconds, minutes, hours or even days depending on the size of the problem as well as the characteristics of our machine. For this simple example, it should be a matter of milliseconds.

In [ ]:
result_type=solver.Solve()

Note that we have stored the result returned by the *Solve* method in a variable called *result_type*. This is important, since as we mentioned in the course, there are multiple types of solutions to a linear optimization problem:

*    A single optimal solution
*    Infinite optimal solutions
*    An unbounded solution
*    An unfeasible problem

*GLOP* informs of the type of solution found with the value returned by the *Solve* method. There are constants defined for each of these states:

| Constante | Estado |
| --- | --- |
| *ABNORMAL*  |  The run finished with an error or problem |
| *FEASIBLE* | Within the time limit specified, if any, we have found a feasible solution |
| *INFEASIBLE* | No feasible solution exists |
| *NOT_SOLVED* | Within the time limit specified, if any, it has found no solution |
| *OPTIMAL*    | The optimal solution has been found |
| *UNBOUNDED*  | The solution is unbounded |

These constants are defined in any instance of the *Solver* object itself. In general, the *FEASIBLE* and *NOT_SOLVED* states can occur when we set some time limit for the solver. When that time runs out we may have a solution (which may not be optimal) or it may have been unable to find any solution.

As a general rule, it is good practice to check the type of result obtained before making any interpretation. In the following code block we make these checks.



In [ ]:
if result_type == solver.ABNORMAL :
  print("An error occurred while executing the solver")
elif result_type == solver.FEASIBLE :
  print("A feasible solution has been found")
  print("The value for variable ", A.name(), 'is', A.solution_value())
  print("The value for variable ", B.name(), 'is', B.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.INFEASIBLE :
  print("The problem has no feasible solution")
elif result_type == solver.NOT_SOLVED :
  print("The solver cound not find any solution so far")
elif result_type == solver.OPTIMAL :
  print("The optimal solution has been found")
  print("The value for variable ", A.name(), 'is', A.solution_value())
  print("The value for variable ", B.name(), 'is', B.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.UNBOUNDED :
  print("The solution is unbounded")
else :
  print("Unknown error code")

There are several things that need to be highlighted the above code snippet. Let's go step by step:

```python
A.solution_value()
```
Note that once the solver is executed, we can obtain the value of the decision variable in the solution using the *solution_value* method

```python
objective.Value()
```
The value of the objective function can be calculated automatically with the *Value* method defined on the objective, after running the solver.

## 8. Summary of the code

The following is all the code developed for the resolution of the previous scenario

In [ ]:
#Creating the solver instance
solver = pywraplp.Solver.CreateSolver('GLOP')

#Creating decision variables
A = solver.NumVar(0, solver.infinity(), name='A')
B = solver.NumVar(0, solver.infinity(), name='B')

#Creating constraints
c_mixing = solver.Constraint(-solver.infinity(), 11, "mixing")
c_mixing.SetCoefficient(A, 2)
c_mixing.SetCoefficient(B, 4)

c_pigmentation = solver.Constraint(-solver.infinity(), 9)
c_pigmentation.SetCoefficient(A, 3)
c_pigmentation.SetCoefficient(B, 1)

#Creating the objective function

objective = solver.Objective()
objective.SetCoefficient(A, 6)
objective.SetCoefficient(B, 2.1)
objective.SetMaximization()

#Solving the problem
result_type=solver.Solve()

if result_type == solver.ABNORMAL :
  print("An error occurred while executing the solver")
elif result_type == solver.FEASIBLE :
  print("A feasible solution has been found")
  print("The value for variable ", A.name(), 'is', A.solution_value())
  print("The value for variable ", B.name(), 'is', B.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.INFEASIBLE :
  print("The problem has no feasible solution")
elif result_type == solver.NOT_SOLVED :
  print("The solver cound not find any solution so far")
elif result_type == solver.OPTIMAL :
  print("The optimal solution has been found")
  print("The value for variable ", A.name(), 'is', A.solution_value())
  print("The value for variable ", B.name(), 'is', B.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.UNBOUNDED :
  print("The solution is unbounded")
else :
  print("Unknown error code")

# Task: Solving and interpreting the solution of linear programming scenarios

Using what you have learned so far, model and solve the following linear programming scenarios. For each of the following scenarios you should:
* Model on paper the scenario
* Program the model using ORTools and GLOP
* Run and interpret the solution to the problem
    * Determine what type of solution is found.
    * If a solution has been found, determine the value of the decision variables in the solution found, as well as the value of the objective function.

## Scenario 1: Marketing investing plan

*A marketing company is going to promote a new gamer headset. To do so, it can invest different amounts of money in three different channels: social media, television and influencers.*

*Previous experiences of the marketing company have established that, on average, for every euro invested in social media advertising there is a return of 2 euros. In the case of television, the return is €1.3 for every €1 invested, while in the case of influencers it is €2.1 per €1 invested.*

*At most, the company has a budget of €2,000 to spend. By a legal contract with the TV networks, the company cannot spend more than 500 euros difference between what is invested in influencers and what is invested on TV. In addition, the company has a pre-contracted minimum spend of 200 euros on social media for each project.*

*How should the marketing company invest?*


In [4]:
from ortools.linear_solver import pywraplp
#Create the solve instance
solver = pywraplp.Solver.CreateSolver('GLOP')

#Create decision variables
SM = solver.NumVar(200, solver.infinity(), name='SM')
TV = solver.NumVar(0, solver.infinity(), name='TV')
I = solver.NumVar(0, solver.infinity(), name='I')

#Create constraints
budget = solver.Constraint(-solver.infinity(), 2000, "budget")
budget.SetCoefficient(SM, 1)
budget.SetCoefficient(TV, 1)
budget.SetCoefficient(I, 1)

difference = solver.Constraint(-solver.infinity(), 500, "difference")
difference.SetCoefficient(SM, 1)
difference.SetCoefficient(TV, -1)

#Create the objective function
objective = solver.Objective()
objective.SetCoefficient(SM, 2.1)
objective.SetCoefficient(TV, 1.3)
objective.SetCoefficient(I, 2)
objective.SetMaximization()


#Solving the problem and interpreting the solution
result_type=solver.Solve()

if result_type == solver.ABNORMAL :
  print("An error occurred while executing the solver")
elif result_type == solver.FEASIBLE :
  print("A feasible solution has been found")
  print("The value for variable ", SM.name(), 'is', SM.solution_value())
  print("The value for variable ", TV.name(), 'is', TV.solution_value())
  print("The value for variable ", I.name(), 'is', I.solution_value())  
  print("The value for the objective function is", objective.Value())
elif result_type == solver.INFEASIBLE :
  print("The problem has no feasible solution")
elif result_type == solver.NOT_SOLVED :
  print("The solver cound not find any solution so far")
elif result_type == solver.OPTIMAL :
  print("The optimal solution has been found")
  print("The value for variable ", SM.name(), 'is', SM.solution_value())
  print("The value for variable ", TV.name(), 'is', TV.solution_value())
  print("The value for variable ", I.name(), 'is', I.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.UNBOUNDED :
  print("The solution is unbounded")
else :
  print("Unknown error code")

The optimal solution has been found
The value for variable  SM is 500.0
The value for variable  TV is 0.0
The value for variable  I is 1500.0
The value for the objective function is 4050.0


## Scenario 2: Mixing in the construction sector

*A construction company requires a minimum of 10,000 cubic meters of sand and gravel mix. The mixture should have no less than 5,000 cubic meters of sand and no more than 6,000 cubic meters of gravel. The materials for this mix can be obtained from two different sites: a material composed by 30% sand and 70% gravel, obtained from site 1 at a cost of €5 per cubic meter, and a material composed by 60% sand and 40% gravel from site 2 at a cost of €7 per cubic meter. Solve this linear programming problem to minimize costs.*

In [5]:
#Create the solve instance
solver = pywraplp.Solver.CreateSolver('GLOP')

#Create decision variables
X1 = solver.NumVar(0, solver.infinity(), name='X1')
X2 = solver.NumVar(0, solver.infinity(), name='X2')

#Create constraints
# Restricción 1: Volumen total >= 10000
# Límite inferior: 10000, Límite superior: infinito
volumen = solver.Constraint(10000, solver.infinity(), "volumen")
volumen.SetCoefficient(X1, 1)
volumen.SetCoefficient(X2, 1)

# Restricción 2: Mínimo de arena 0.30*X1 + 0.60*X2 >= 5000
# Límite inferior: 5000, Límite superior: infinito
arena = solver.Constraint(5000, solver.infinity(), "arena")
arena.SetCoefficient(X1, 0.30)
arena.SetCoefficient(X2, 0.60)

# Restricción 3: Máximo de grava 0.70*X1 + 0.40*X2 <= 6000
# Límite inferior: -infinito, Límite superior: 6000
grava = solver.Constraint(-solver.infinity(), 6000, "grava")
grava.SetCoefficient(X1, 0.70)
grava.SetCoefficient(X2, 0.40)

#Create the objective function
objective = solver.Objective()
objective.SetCoefficient(X1, 5)
objective.SetCoefficient(X2, 7)
objective.SetMinimization()

#Solving the problem and interpreting the solution
result_type=solver.Solve()

if result_type == solver.ABNORMAL :
  print("An error occurred while executing the solver")
elif result_type == solver.FEASIBLE :
  print("A feasible solution has been found")
  print("The value for variable ", X1.name(), 'is', X1.solution_value())
  print("The value for variable ", X2.name(), 'is', X2.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.INFEASIBLE :
  print("The problem has no feasible solution")
elif result_type == solver.NOT_SOLVED :
  print("The solver cound not find any solution so far")
elif result_type == solver.OPTIMAL :
  print("The optimal solution has been found")
  print("The value for variable ", X1.name(), 'is', X1.solution_value())
  print("The value for variable ", X2.name(), 'is', X2.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.UNBOUNDED :
  print("The solution is unbounded")
else :
  print("Unknown error code")


The optimal solution has been found
The value for variable  X1 is 3333.333333333334
The value for variable  X2 is 6666.666666666666
The value for the objective function is 63333.333333333336


## Scenario 3: A deployment plan in the cloud

*A software company deploys and maintains two applications in the cloud. Each month it has a maximum expenditure for cloud resources of 1500 euros. For each application, both disk and computational capacity can be spent.*

*The cost of each MB of storage for an application is 0.01 euros, while each second of processing per application costs 0.05 euros. It has been estimated that in a typical month of operation, the first application requires a minimum of 100 MB of storage and 10 hours of processing. On the other hand, the second application requires a minimum of 200 MB of storage and 5 hours of processing.*

*Both applications are related, so they must be scaled accordingly. For every MB allocated to the first application, there will be a minimum of 3MB allocated to the second application. Similarly, for every second of computation in the first application, 2 seconds of computation are required in the second application.*

*Determine an optimal cloud spending plan so that the requirements of both applications are met.*

In [ ]:
#Create the solve instance
solver = pywraplp.Solver.CreateSolver('GLOP')

#Create decision variables
X1 = solver.NumVar(100, solver.infinity(), name='X1')
X2 = solver.NumVar(200, solver.infinity(), name='X2')
T1 = solver.NumVar(36000, solver.infinity(), name='T1')
T2 = solver.NumVar(18000, solver.infinity(), name='T2')

#Create constraints
# Restricción 1: X1 + X2 + T1 + T2 <= 1500
presupuesto = solver.Constraint(-solver.infinity(), 1500, "constraint1") # ¡¡Se puede eliminar el limite superior para ver q presupuesto sería necesario!!
presupuesto.SetCoefficient(X1, 1)
presupuesto.SetCoefficient(X2, 1)
presupuesto.SetCoefficient(T1, 1)
presupuesto.SetCoefficient(T2, 1)

# Restricción 2: MB (-3 X1 + X2 >= 0)
MB = solver.Constraint(0, solver.infinity(), "constraint2")
MB.SetCoefficient(X1, -3)
MB.SetCoefficient(X2, 1)

# Restricción 3: Escalado de procesamiento (-2X1 + X2 >= 0)
tiempo = solver.Constraint(0, solver.infinity(), "constraint3")
tiempo.SetCoefficient(X1, -2)
tiempo.SetCoefficient(X2, 1)

#Create the objective function
objective = solver.Objective()
objective.SetCoefficient(X1, 0.01)
objective.SetCoefficient(X2, 0.01)
objective.SetCoefficient(T1, 0.05)
objective.SetCoefficient(T2, 0.05)
objective.SetMinimization()

#Solving the problem and interpreting the solution
result_type = solver.Solve()
if result_type == solver.ABNORMAL :
  print("An error occurred while executing the solver")
elif result_type == solver.FEASIBLE :
  print("A feasible solution has been found")
  print("The value for variable ", X1.name(), 'is', X1.solution_value())
  print("The value for variable ", X2.name(), 'is', X2.solution_value())
  print("The value for variable ", T1.name(), 'is', T1.solution_value())
  print("The value for variable ", T2.name(), 'is', T2.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.INFEASIBLE :
  print("The problem has no feasible solution")
elif result_type == solver.NOT_SOLVED :
  print("The solver cound not find any solution so far")
elif result_type == solver.OPTIMAL :
  print("The optimal solution has been found")
  print("The value for variable ", X1.name(), 'is', X1.solution_value())
  print("The value for variable ", X2.name(), 'is', X2.solution_value())
  print("The value for variable ", T1.name(), 'is', T1.solution_value())
  print("The value for variable ", T2.name(), 'is', T2.solution_value())
  print("The value for the objective function is", objective.Value())
elif result_type == solver.UNBOUNDED :
  print("The solution is unbounded")
else :
  print("Unknown error code")

The problem has no feasible solution
